# 02 — Evaluation Setup

This notebook defines, and then *uses only for model selection*, the full
evaluation pipeline: the temporal train/validation/test split, the
candidate/exclusion policy, the metrics, and a validation-only hyperparameter
search for every model. **The test set is not touched anywhere in this
notebook** — it is reserved entirely for the one-time final evaluation in
`03_models.ipynb`.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking

from src import data as D
from src import evaluation as E
from src import metrics as M
from src.recommenders import (
    PopularityRecommender, ItemKNNRecommender, ContentBasedRecommender,
    FactorModelRecommender, HybridRecommender,
)

K = 10
pd.set_option("display.max_columns", 50)

## 1. Load data and build the temporal split

For each user, interactions are sorted by timestamp and cut into three
contiguous blocks: the earliest ~70% go to **train**, the next ~15% to
**validation**, and the most recent ~15% to **test**. Every user contributes
to all three blocks because MovieLens-1M guarantees at least 20 ratings per
user (`01_data_and_eda.ipynb`).

- **train** — fit every model.
- **validation** — choose hyperparameters, the content-history threshold, and
  the hybrid weight (this notebook).
- **test** — touched exactly once, for the final numbers reported in
  `03_models.ipynb` and `04_product_analysis.ipynb`.

In [2]:
users, movies, ratings = D.load_raw("../data")
ratings, user_id_to_idx, item_id_to_idx = D.add_index_columns(ratings)
df = D.temporal_split(ratings)
n_users = df["user_idx"].nunique()

print(df["split"].value_counts())
print((df["split"].value_counts(normalize=True) * 100).round(1))

split
train    700022
test     150177
val      150010
Name: count, dtype: int64
split
train    70.0
test     15.0
val      15.0
Name: proportion, dtype: float64


## 2. Candidate universe and the already-seen exclusion rule

Every model ranks the same candidate set: **items observed at least once in
the training data available at that point** (train only, while tuning on
validation; train+validation, for the final test evaluation in `03`). Movies
that only appear later are "item cold start" and are excluded from every
model's candidate set alike (see `01_data_and_eda.ipynb`, ~7.8% of movies
have fewer than 5 ratings).

For every user, all items they have *already interacted with* (any rating,
not just relevant ones — a user cannot be usefully re-shown something they
already watched) are removed from their candidate list before ranking.

In [3]:
catalog_items = D.catalog_from_split(df, ["train"])
item_pos = E.item_position_map(catalog_items)
train_df = df[df["split"] == "train"]
train_df = train_df[train_df["item_idx"].isin(item_pos)]

seen = D.get_seen(df, ["train"])
truth = D.get_truth(df, "val", threshold=D.RELEVANT_RATING_THRESHOLD)
pop_count = E.item_pop_count(df, ["train"])
feature_matrix, genre_names = D.build_movie_features(movies, catalog_items, item_id_to_idx)

users_to_eval = list(truth.keys())
print(f"Candidate catalog: {len(catalog_items)} movies")
print(f"Validation evaluation cohort: {len(users_to_eval)} / {n_users} users "
      f"({len(users_to_eval)/n_users:.1%}) have >= 1 relevant (rating>=4) item in validation")

Candidate catalog: 3649 movies
Validation evaluation cohort: 5920 / 6040 users (98.0%) have >= 1 relevant (rating>=4) item in validation


Note the ~2% of users excluded from the cohort: they had no `rating >= 4`
interaction in their validation slice (mostly users who happened to rate
everything in that window below 4). Recall/Precision/MAP/NDCG are undefined
without at least one relevant item, so they are excluded from the averaged
metric rather than silently counted as zero — the same rule is applied
identically to every model.

## 3. Metrics

All ranking metrics are computed at **K = 10**.

- **Recall@10** — share of a user's relevant items that appear in the top 10.
- **Precision@10** — share of the top 10 that are relevant.
- **MAP@10** — average precision at each rank a relevant item is hit, rewarding relevant items ranked higher.
- **NDCG@10** — like MAP, but discounts by `1/log2(rank+1)` and normalizes against the best possible ordering; the primary metric used for model selection here because it directly rewards ranking quality rather than just retrieval.

Beyond-accuracy metrics (computed in `03_models.ipynb` on the final test predictions):

- **Catalog Coverage@10** — share of the whole catalog that appears in *any* user's top 10, across all users. Low coverage means the system effectively only ever recommends a small subset of the catalog.
- **Novelty@10** — mean self-information `-log2(popularity share)` of recommended items; higher means less-popular ("long tail") items are being surfaced.
- **Intra-list Diversity@10** — mean pairwise `(1 - cosine similarity)` between the genre/year vectors of items *within a single user's* list; higher means a single list spans more different kinds of movies.
- **Average Recommended Popularity@10** — mean training-set interaction count of recommended items, a simple, literal popularity-bias indicator.

Implementations live in `src/metrics.py`.

## 4. Model selection on validation

Every model below is fit on **train only** and scored against the
**validation** relevant-item truth, using the shared candidate/exclusion
policy from Section 2. The hyperparameter (or design choice) with the
highest validation NDCG@10 is kept; it is hard-coded into
`src/pipeline.py::BEST_PARAMS` and reused, without modification, on the
never-before-seen test set in `03_models.ipynb`.

### 4.1 Popularity baseline (no hyperparameters)

In [4]:
pop_model = PopularityRecommender().fit(catalog_items, pop_count)
rec_pop = E.generate_rec_lists(pop_model, users_to_eval, seen, catalog_items, item_pos, K)
print(M.evaluate_recommendations(rec_pop, truth, K))

{'Recall@K': 0.051030875926588164, 'Precision@K': 0.06363175675675677, 'MAP@K': 0.03551204512867459, 'NDCG@K': 0.07590711249503716, 'n_users': 5920}


### 4.2 Item-kNN (no hyperparameters beyond similarity choice)

In [5]:
knn_model = ItemKNNRecommender().fit(train_df, catalog_items, item_pos)
rec_knn = E.generate_rec_lists(knn_model, users_to_eval, seen, catalog_items, item_pos, K)
print(M.evaluate_recommendations(rec_knn, truth, K))

{'Recall@K': 0.09301606710185703, 'Precision@K': 0.0860641891891892, 'MAP@K': 0.05466176359182311, 'NDCG@K': 0.11135094859624498, 'n_users': 5920}


### 4.3 Content-based: choosing the history threshold

The content-based user profile is built from movies in a user's *train*
history rated at or above `history_min_rating`. Using every interaction
(including disliked movies) to build the taste profile is questionable in
the same way as the relevance definition itself — so this threshold is
tuned on validation rather than assumed.

In [6]:
cb_results = {}
for min_rating in [None, 3, 4]:
    cb_model = ContentBasedRecommender().fit(
        train_df, catalog_items, item_pos, feature_matrix, min_rating, pop_model.scores
    )
    rec_cb = E.generate_rec_lists(cb_model, users_to_eval, seen, catalog_items, item_pos, K)
    m = M.evaluate_recommendations(rec_cb, truth, K)
    cb_results[min_rating] = (m, cb_model)
    print(f"min_rating={min_rating}: {m}")

best_min_rating = max(cb_results, key=lambda mr: cb_results[mr][0]['NDCG@K'])
print(f"\nChosen history_min_rating = {best_min_rating}")

min_rating=None: {'Recall@K': 0.015673257720260944, 'Precision@K': 0.012449324324324326, 'MAP@K': 0.0070656658465438225, 'NDCG@K': 0.01670002419054012, 'n_users': 5920}


min_rating=3: {'Recall@K': 0.016315425914726556, 'Precision@K': 0.012736486486486488, 'MAP@K': 0.007021946797988465, 'NDCG@K': 0.01699487589758238, 'n_users': 5920}


min_rating=4: {'Recall@K': 0.015786966105725324, 'Precision@K': 0.012820945945945947, 'MAP@K': 0.007095751071011488, 'NDCG@K': 0.017195966460638753, 'n_users': 5920}

Chosen history_min_rating = 4


### 4.4 ALS — and the orientation bug found during the audit

**Finding from the audit of the previous version of this project:** the ALS
model was fit as `als_model.fit(train_matrix)` where `train_matrix` had
shape `(n_items, n_users)`. The `implicit` library's `AlternatingLeastSquares.fit`
expects a `(n_users, n_items)` matrix — the matrix was transposed. This
silently swaps the meaning of "user factors" and "item factors": row `u` of
`user_factors` ends up describing *item* `u`, not user `u`. Every downstream
recommendation was therefore scored against the wrong entity, which is the
real reason the old notebook's implicit-ALS evaluation collapsed to exactly
zero — not evidence that "ALS doesn't work for temporal splits".

The cell below reproduces the bug in isolation, on this project's own data
and metrics, to confirm the diagnosis before fixing it.

In [7]:
# --- Reproduce the orientation bug for the audit record ---
from scipy.sparse import csr_matrix

wrong_rows = np.array([item_pos[i] for i in train_df["item_idx"].values])
wrong_cols = train_df["user_idx"].values
wrong_vals = 1.0 + 2.0 * (train_df["rating"].values.astype(float) - 1.0)
wrong_matrix = csr_matrix((wrong_vals, (wrong_rows, wrong_cols)), shape=(len(catalog_items), n_users))

buggy_model = AlternatingLeastSquares(factors=64, regularization=0.1, iterations=20, random_state=42)
buggy_model.fit(wrong_matrix, show_progress=False)  # fit on (items, users) -- WRONG orientation

print("fit() was called on a (n_catalog, n_users) matrix instead of (n_users, n_catalog).")
print(f"  wrong_matrix.shape = {wrong_matrix.shape}  (rows are treated as 'users' by the library)")
print(f"  user_factors.shape = {buggy_model.user_factors.shape}  <- indexed by CATALOG POSITION, not user_idx")
print(f"  item_factors.shape = {buggy_model.item_factors.shape}  <- indexed by USER_IDX, not catalog position")
print()
print("recommend(userid=user_idx, ...) would look up user_factors[user_idx] -- "
      "but that row describes catalog item `user_idx`, not user `user_idx`. "
      "Because n_users (6040) > n_catalog (3649), most indices stay in-bounds, "
      "so the call does not raise an error: it silently returns a ranking with "
      "no relationship to the target user, which is why the original "
      "notebook's implicit-ALS metrics came out at (or near) zero.")

fit() was called on a (n_catalog, n_users) matrix instead of (n_users, n_catalog).
  wrong_matrix.shape = (3649, 6040)  (rows are treated as 'users' by the library)
  user_factors.shape = (3649, 64)  <- indexed by CATALOG POSITION, not user_idx
  item_factors.shape = (6040, 64)  <- indexed by USER_IDX, not catalog position

recommend(userid=user_idx, ...) would look up user_factors[user_idx] -- but that row describes catalog item `user_idx`, not user `user_idx`. Because n_users (6040) > n_catalog (3649), most indices stay in-bounds, so the call does not raise an error: it silently returns a ranking with no relationship to the target user, which is why the original notebook's implicit-ALS metrics came out at (or near) zero.


In [8]:
# --- Correct orientation: (n_users, n_catalog), confidence-weighted ---
conf_matrix = E.build_confidence_matrix(df, ["train"], n_users, catalog_items, item_pos, alpha=2.0)

als_results = {}
for factors in [32, 64, 128]:
    for reg in [0.01, 0.1]:
        model = AlternatingLeastSquares(factors=factors, regularization=reg, iterations=20, random_state=42)
        als = FactorModelRecommender(model).fit(conf_matrix)
        rec_als = E.generate_rec_lists(als, users_to_eval, seen, catalog_items, item_pos, K)
        m = M.evaluate_recommendations(rec_als, truth, K)
        als_results[(factors, reg)] = (m, als)
        print(f"factors={factors} reg={reg}: {m}")

best_als_params = max(als_results, key=lambda p: als_results[p][0]['NDCG@K'])
print(f"\nChosen (factors, regularization) = {best_als_params}")

factors=32 reg=0.01: {'Recall@K': 0.08648405127859554, 'Precision@K': 0.06503378378378379, 'MAP@K': 0.0413139754337671, 'NDCG@K': 0.08804838383835858, 'n_users': 5920}


factors=32 reg=0.1: {'Recall@K': 0.08601540166525373, 'Precision@K': 0.06483108108108109, 'MAP@K': 0.041116505137152454, 'NDCG@K': 0.08782866647245112, 'n_users': 5920}


factors=64 reg=0.01: {'Recall@K': 0.08717950950880152, 'Precision@K': 0.06339527027027027, 'MAP@K': 0.040242401361821, 'NDCG@K': 0.0866414729814958, 'n_users': 5920}


factors=64 reg=0.1: {'Recall@K': 0.08691597448534608, 'Precision@K': 0.06314189189189189, 'MAP@K': 0.04001043015272926, 'NDCG@K': 0.08628277907304718, 'n_users': 5920}


factors=128 reg=0.01: {'Recall@K': 0.087952158248412, 'Precision@K': 0.06108108108108108, 'MAP@K': 0.040287452578003174, 'NDCG@K': 0.08607405750101525, 'n_users': 5920}


factors=128 reg=0.1: {'Recall@K': 0.08819286287764888, 'Precision@K': 0.06123310810810811, 'MAP@K': 0.04029032441774257, 'NDCG@K': 0.08610783721626, 'n_users': 5920}

Chosen (factors, regularization) = (32, 0.01)


Correcting the orientation (the grid search two cells below, fit on a proper
`(n_users, n_catalog)` matrix) takes NDCG@10 from this silently-broken ~0 up
to roughly **0.09** — comparable to Item-kNN. ALS is a legitimate,
competitive model on this data once the bug is fixed; the earlier "ALS
doesn't suit a temporal split" conclusion in the previous version of this
project was an artifact of a matrix-shape bug, not a property of the
algorithm. This is exactly the kind of mistake the audit was asked to catch
rather than take at face value.

### 4.5 A ranking-loss factorization model in place of LightFM

The brief calls for a hybrid built with **LightFM** (collaborative +
genre item features, WARP/BPR loss). LightFM's PyPI source distribution
does not build on this environment (Windows, Python 3.12): its `setup.py`
predates modern `setuptools` and fails at the metadata-generation step
(`AttributeError: 'dict' object has no attribute '__LIGHTFM_SETUP__'`)
regardless of build-isolation settings, and the project has not published a
compatible release since 2020. Rather than lose time fighting an unmaintained
build toolchain, we substitute **Bayesian Personalized Ranking (BPR)** from
the `implicit` library already used for ALS: it optimizes a pairwise ranking
loss (the same family LightFM's WARP/BPR losses belong to) directly from
implicit feedback, which is the property we actually wanted from LightFM.
It does not consume genre features directly — that gap is covered by
hybridizing it with the content-based model in Section 4.6.

In [9]:
bpr_results = {}
for factors in [32, 64, 128]:
    model = BayesianPersonalizedRanking(factors=factors, iterations=100, random_state=42)
    bpr = FactorModelRecommender(model).fit(conf_matrix)
    rec_bpr = E.generate_rec_lists(bpr, users_to_eval, seen, catalog_items, item_pos, K)
    m = M.evaluate_recommendations(rec_bpr, truth, K)
    bpr_results[factors] = (m, bpr)
    print(f"factors={factors}: {m}")

best_bpr_factors = max(bpr_results, key=lambda f: bpr_results[f][0]['NDCG@K'])
print(f"\nChosen BPR factors = {best_bpr_factors}")

factors=32: {'Recall@K': 0.06454041001599554, 'Precision@K': 0.04481418918918919, 'MAP@K': 0.030387842668348625, 'NDCG@K': 0.06479672572219448, 'n_users': 5920}


factors=64: {'Recall@K': 0.06442879678343322, 'Precision@K': 0.043800675675675665, 'MAP@K': 0.029896313910636827, 'NDCG@K': 0.06371447231064518, 'n_users': 5920}


factors=128: {'Recall@K': 0.06541060842127513, 'Precision@K': 0.04429054054054054, 'MAP@K': 0.03162669703610924, 'NDCG@K': 0.06526704415528461, 'n_users': 5920}

Chosen BPR factors = 128


### 4.6 Hybrid: which collaborative signal pairs best with content?

The hybrid combines the content-based score with one collaborative model's
score (each min-max normalized per user, then blended by weight `alpha`).
Rather than assume a partner and a 50/50 weight, both the partner model and
`alpha` are chosen on validation.

In [10]:
best_als_model = als_results[best_als_params][1]
best_bpr_model = bpr_results[best_bpr_factors][1]
best_cb_model = cb_results[best_min_rating][1]

hybrid_search = {}
for partner_name, partner_model in [("BPR", best_bpr_model), ("ALS", best_als_model), ("Item-kNN", knn_model)]:
    for alpha in np.round(np.arange(0.0, 1.01, 0.1), 2):
        hyb = HybridRecommender(best_cb_model, partner_model, alpha)
        rec_hyb = E.generate_rec_lists(hyb, users_to_eval, seen, catalog_items, item_pos, K)
        m = M.evaluate_recommendations(rec_hyb, truth, K)
        hybrid_search[(partner_name, alpha)] = m['NDCG@K']

best_partner, best_alpha = max(hybrid_search, key=hybrid_search.get)
print(f"Best pairing: content-based + {best_partner}, alpha={best_alpha} "
      f"(NDCG@10={hybrid_search[(best_partner, best_alpha)]:.5f})")

import pandas as pd
pd.Series(hybrid_search).unstack(0).round(4)

Best pairing: content-based + Item-kNN, alpha=0.1 (NDCG@10=0.11170)


,ALS,BPR,Item-kNN
0.0,0.0880,0.0653,0.1114
0.1,0.0894,0.0661,0.1117
0.2,0.0913,0.0660,0.1082
0.3,0.0914,0.0653,0.1026
0.4,0.0899,0.0625,0.0966
0.5,0.0871,0.0588,0.0902
0.6,0.0826,0.0541,0.0826
0.7,0.0754,0.0484,0.0725
0.8,0.0640,0.0429,0.0619
0.9,0.0495,0.0351,0.0488


`alpha` here is the content-based model's weight: `alpha=0.0` is pure
collaborative, `alpha=1.0` is pure content-based. The best pairing is a
**small** content-based contribution (`alpha≈0.1`) added to the strongest
collaborative model on this data — content alone is too weak to carry the
combination, but a modest blend nudges the collaborative model's ranking.

## 5. Chosen hyperparameters

In [11]:
BEST_PARAMS = {
    "content_min_rating": best_min_rating,
    "als_factors": best_als_params[0],
    "als_regularization": best_als_params[1],
    "bpr_factors": best_bpr_factors,
    "hybrid_partner": best_partner,
    "hybrid_alpha": best_alpha,
}
BEST_PARAMS

{'content_min_rating': 4,
 'als_factors': 32,
 'als_regularization': 0.01,
 'bpr_factors': 128,
 'hybrid_partner': 'Item-kNN',
 'hybrid_alpha': np.float64(0.1)}

These values match `src/pipeline.py::BEST_PARAMS`, used as-is (no further
tuning) in `03_models.ipynb` to fit final models on train+validation and
evaluate once on test.